# EDA de reclamos CFPB

Este notebook documenta el análisis exploratorio paso a paso. Cada etapa se revisa antes de continuar.

## E0 — Verificar el archivo raw

**Pregunta:** ¿Estamos trabajando con el archivo correcto y cuál es su estructura física?

En este paso no transformamos datos ni cargamos el corpus completo en memoria. Solo revisamos el archivo, sus metadatos Parquet, su esquema y cinco registros protegidos.

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq
from IPython.display import display

### 1. Localizar el raw y su puntero DVC

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / "data/raw/cfpb_reclamos_narrativa.parquet"
DVC_PATH = PROJECT_ROOT / "data/raw/cfpb_reclamos_narrativa.parquet.dvc"

assert RAW_PATH.exists(), f"No se encontró el raw: {RAW_PATH}"
assert DVC_PATH.exists(), f"No se encontró el puntero DVC: {DVC_PATH}"

dvc_lines = DVC_PATH.read_text(encoding="utf-8").splitlines()
dvc_md5 = next(line.split(":", 1)[1].strip() for line in dvc_lines if line.strip().startswith("- md5:"))

display(pd.DataFrame({
    "campo": ["archivo", "tamaño físico", "MD5 registrado por DVC"],
    "valor": [RAW_PATH.name, f"{RAW_PATH.stat().st_size / 1024**3:.2f} GiB", dvc_md5],
}))

,campo,valor
0,archivo,cfpb_reclamos_narrativa.parquet
1,tamaño físico,1.51 GiB
2,MD5 registrado por DVC,68f4560aca0967779f118e8225e72c26


### 2. Leer metadatos Parquet

Los metadatos permiten contar filas y columnas sin cargar todas las narrativas.

In [3]:
parquet = pq.ParquetFile(RAW_PATH)
metadata = parquet.metadata
logical_bytes = sum(metadata.row_group(i).total_byte_size for i in range(metadata.num_row_groups))

summary = pd.DataFrame({
    "medida": [
        "filas",
        "columnas",
        "grupos de filas",
        "tamaño físico",
        "tamaño lógico Parquet",
        "creado por",
    ],
    "valor": [
        f"{metadata.num_rows:,}",
        metadata.num_columns,
        metadata.num_row_groups,
        f"{RAW_PATH.stat().st_size / 1024**3:.2f} GiB",
        f"{logical_bytes / 1024**3:.2f} GiB",
        metadata.created_by,
    ],
})

assert metadata.num_rows == 3_837_184
assert metadata.num_columns == 16
display(summary)

,medida,valor
0,filas,"3,837,184"
1,columnas,16
2,grupos de filas,83
3,tamaño físico,1.51 GiB
4,tamaño lógico Parquet,3.73 GiB
5,creado por,parquet-cpp-arrow version 24.0.0


### 3. Revisar el esquema físico

In [4]:
schema = parquet.schema_arrow
schema_table = pd.DataFrame({
    "columna": [field.name for field in schema],
    "tipo parquet": [str(field.type) for field in schema],
    "permite nulos": [field.nullable for field in schema],
})
display(schema_table)

,columna,tipo parquet,permite nulos
0,Date received,large_string,True
1,Product,large_string,True
2,Sub-product,large_string,True
3,Issue,large_string,True
4,Sub-issue,large_string,True
5,Consumer complaint narrative,large_string,True
6,Company public response,large_string,True
7,Company,large_string,True
8,State,large_string,True
9,ZIP code,large_string,True


### 4. Verificar cinco registros sin publicar narrativas

La narrativa se reemplaza por su cantidad de caracteres para no guardar texto de consumidores en la salida del notebook.

In [5]:
sample = ds.dataset(RAW_PATH, format="parquet").head(5).to_pandas()
narrative_column = "Consumer complaint narrative"
sample[narrative_column] = sample[narrative_column].map(
    lambda text: f"<narrativa: {len(text):,} caracteres>" if isinstance(text, str) else "<sin narrativa>"
)
display(sample.T)

,0,1,2,3,4
Date received,2023-04-21,2023-07-13,2023-06-27,2024-05-06,2025-06-25
Product,Checking or savings account,"Money transfer, virtual currency, or money ser...",Mortgage,Debt collection,Student loan
Sub-product,Checking account,International money transfer,Other type of mortgage,Auto debt,Federal student loan servicing
Issue,Managing an account,Fraud or scam,Trouble during payment process,Took or threatened to take negative or legal a...,Dealing with your lender or servicer
Sub-issue,Deposits and withdrawals,NaN,NaN,Seized or attempted to seize your property,Trouble with how payments are being handled
Consumer complaint narrative,"<narrativa: 1,094 caracteres>","<narrativa: 1,392 caracteres>","<narrativa: 10,703 caracteres>",<narrativa: 779 caracteres>,"<narrativa: 1,206 caracteres>"
Company public response,Company has responded to the consumer and the ...,Company has responded to the consumer and the ...,Company has responded to the consumer and the ...,NaN,NaN
Company,U.S. BANCORP,"BANK OF AMERICA, NATIONAL ASSOCIATION",Specialized Loan Servicing Holdings LLC,TOYOTA MOTOR CREDIT CORPORATION,MOHELA
State,CA,GA,CA,CA,MA
ZIP code,91362,30032,94954,94112,02360


## Resultado de E0

- El archivo raw y su puntero DVC están disponibles.
- El archivo contiene **3,837,184 filas y 16 columnas**.
- La estructura puede inspeccionarse sin cargar el corpus completo en memoria.
- E0 no modifica el raw ni genera datos derivados.
- El siguiente paso, sujeto a aprobación, es **E1: revisar tipos, nulos, dominios y ejemplos de cada columna**.